# VZLA Sports Elite — ML Starter Notebook

This notebook shows how to get started with the auto-updating `data/ml-dataset.csv`.

We cover two quick baselines:
1. **Volatility clustering** — group athletes by how stable or jumpy their prices are.
2. **7-day price-direction classifier** — predict whether an athlete's raw price will be higher one week from now.

Run this locally or in Google Colab. To use the live dataset from GitHub, replace the path below with the raw URL.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Adjust path if running outside the repo root
CSV_PATH = Path("../data/ml-dataset.csv")

df = pd.read_csv(CSV_PATH, parse_dates=["date"])
df = df.sort_values(["name", "date"]).reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"Athletes: {df['name'].nunique():,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 1. Quick exploration

In [ ]:
# Missing-value overview
missing = df.isna().mean().sort_values(ascending=False) * 100
missing[missing > 0].plot(kind="barh", figsize=(8, 5), title="Missing values (% of rows)")
plt.show()

# Sport breakdown
df.groupby("sport").agg(
    athletes=("name", "nunique"),
    avg_price=("raw_price", "mean"),
    median_cv=("raw_cv", "median"),
).round(3)

## 2. Volatility clustering baseline

Cluster athletes by recent price behavior. We standardize within each sport because Baseball and Soccer operate on very different price scales.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Use the most recent row per athlete for a snapshot clustering
latest = df.groupby("name").last().reset_index()

cluster_cols = ["raw_cv", "raw_return_vol_7d", "raw_price_chg_7d_pct", "raw_price_chg_30d_pct"]
model_df = latest.dropna(subset=cluster_cols + ["sport"]).copy()

# Standardize within sport so clustering is about shape, not absolute price
features = []
for sport, group in model_df.groupby("sport"):
    if len(group) < 10:
        continue
    scaler = StandardScaler()
    scaled = scaler.fit_transform(group[cluster_cols])
    group = group.copy()
    for i, col in enumerate(cluster_cols):
        group[f"z_{col}"] = scaled[:, i]
    features.append(group)

features = pd.concat(features, ignore_index=True)
z_cols = [f"z_{c}" for c in cluster_cols]

# K-Means with 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
features["cluster"] = kmeans.fit_predict(features[z_cols])

# Show cluster profiles
profiles = features.groupby("cluster")[cluster_cols].mean().round(3)
print(profiles)
print("\nCluster sizes:")
print(features["cluster"].value_counts().sort_index())

In [ ]:
# Visualize clusters in 2D
pca = PCA(n_components=2)
features[["pca_1", "pca_2"]] = pca.fit_transform(features[z_cols])

plt.figure(figsize=(9, 6))
for c in sorted(features["cluster"].unique()):
    subset = features[features["cluster"] == c]
    plt.scatter(subset["pca_1"], subset["pca_2"], label=f"Cluster {c}", alpha=0.6, s=40)
plt.title("Athlete volatility clusters (PCA projection)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend()
plt.show()

# Example athletes from each cluster
for c in sorted(features["cluster"].unique()):
    print(f"\nCluster {c} examples:")
    print(features[features["cluster"] == c][["name", "sport", "raw_price", "raw_cv"]].head(5).to_string(index=False))

## 3. 7-day price-direction classifier

Target: is `raw_price` higher 7 days from now than it is today?

**Critical rule:** only use features known on or before `date`. No future leakage.

In [ ]:
# Build point-in-time features per athlete
def add_rolling_features(group):
    group = group.sort_values("date")
    group["raw_price_lag_7"] = group["raw_price"].shift(7)
    group["raw_price_lag_14"] = group["raw_price"].shift(14)
    group["raw_price_lag_30"] = group["raw_price"].shift(30)
    group["raw_price_ma_7"] = group["raw_price"].rolling(7, min_periods=3).mean()
    group["raw_price_ma_30"] = group["raw_price"].rolling(30, min_periods=5).mean()
    group["raw_volume_ma_7"] = group["raw_n_listings"].rolling(7, min_periods=3).mean()
    group["target_up_7d"] = (group["raw_price"].shift(-7) > group["raw_price"]).astype(int)
    return group

frames = [add_rolling_features(group) for _, group in df.groupby("name")]
df_model = pd.concat(frames, ignore_index=True)

feature_cols = [
    "raw_price", "raw_cv", "raw_n_listings", "raw_index",
    "raw_price_chg_7d_pct", "raw_price_chg_30d_pct", "raw_return_vol_7d",
    "days_on_market", "days_since_first_seen",
    "raw_price_lag_7", "raw_price_lag_14", "raw_price_lag_30",
    "raw_price_ma_7", "raw_price_ma_30", "raw_volume_ma_7",
]

# Drop rows without a target or with missing core features
train_df = df_model.dropna(subset=["target_up_7d"] + feature_cols).copy()

print(f"Model rows: {len(train_df):,}")
print(f"Baseline accuracy (always predict 'up'): {train_df['target_up_7d'].mean():.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Time-series split: train on older data, test on newer data
split_date = train_df["date"].quantile(0.8)
train = train_df[train_df["date"] < split_date]
test = train_df[train_df["date"] >= split_date]

X_train, y_train = train[feature_cols], train["target_up_7d"]
X_test, y_test = test[feature_cols], test["target_up_7d"]

# Logistic regression baseline
lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
print("Logistic Regression:")
print(classification_report(y_test, lr_pred, digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1]):.3f}")

# Random forest (no hyperparameter tuning yet)
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("\nRandom Forest:")
print(classification_report(y_test, rf_pred, digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]):.3f}")

In [ ]:
# Feature importance from the random forest
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
importance.plot(kind="barh", figsize=(8, 6), title="Feature importance (Random Forest)")
plt.show()

## 4. Group-aware time-series validation (more realistic)

A stronger check: make sure the same athlete never appears in both train and test. This tells you whether the model generalizes to athletes it has not seen.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(train_df, groups=train_df["name"]))

X_train_g, y_train_g = train_df.iloc[train_idx][feature_cols], train_df.iloc[train_idx]["target_up_7d"]
X_test_g, y_test_g = train_df.iloc[test_idx][feature_cols], train_df.iloc[test_idx]["target_up_7d"]

rf_g = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1)
rf_g.fit(X_train_g, y_train_g)
print("Group-aware Random Forest:")
print(classification_report(y_test_g, rf_g.predict(X_test_g), digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test_g, rf_g.predict_proba(X_test_g)[:, 1]):.3f}")

## 5. Export a sample prediction CSV

Use the trained model to score the most recent observation for each athlete. This is the same format you could later feed back into the app.

In [ ]:
latest_scored = train_df.groupby("name").last().reset_index().dropna(subset=feature_cols).copy()
latest_scored["pred_prob_up_7d"] = rf.predict_proba(latest_scored[feature_cols])[:, 1]

output = latest_scored[["name", "sport", "date", "raw_price", "pred_prob_up_7d"]].sort_values(
    "pred_prob_up_7d", ascending=False
)

output.to_csv("ml-predictions-sample.csv", index=False)
print("Top 10 highest predicted probability of price increase:")
print(output.head(10).to_string(index=False))

## Next steps

- Try XGBoost or LightGBM for the classifier.
- Add athlete-level static features (`psa_pop`, `scp_raw_price`) as descriptors, but remember they are snapshot-joined, not historical.
- Build a "deal score" by combining predicted upside with current price and listing volume.
- Once you trust a model, we can add a page to the app that surfaces the top predictions.